In [ ]:
import os
os.chdir(path_to_wd)

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import torch
print(torch.cuda.is_available()) 
import scvi
import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.colors import ListedColormap
import sys
import scipy
import scipy.sparse as sp
from scipy.sparse import csr_matrix, csc_matrix, coo_matrix, lil_matrix
from scipy import io
from scipy.io import mmread
import pycats
from pandas.api.types import CategoricalDtype
import hotspot
import seaborn as sns

scvi.settings.seed = 12345
plt.rcParams['font.family']=['Dejavu serif']
plt.rcParams['figure.dpi'] = 100
plt.rcParams['pdf.fonttype'] = 'truetype'

In [ ]:
# color map
blue_yellow_colors = ["#352A86","#343DAE","#0262E0","#1389D2","#2DB7A3","#A5BE6A","#F8BA43","#F6DA23","#F8FA0D"]
horizon_colors = ["#000033","#000075","#0000B6","#0000F8","#2E00FF","#6100FF","#9408F7","#C729D6","#FA4AB5",
                 "#FF6A95","#FF8B74","#FFAC53","#FFCD32","#FFEE11","#FFFF60"]
solar_extra_colors = ["#3361A5","#248AF3","#14B3FF","#88CEEF","#C1D5DC","#EAD397","#FDB31A","#E42A2A","#A31D1D"]

blue_yellow = LinearSegmentedColormap.from_list('custom',blue_yellow_colors)
horizon = LinearSegmentedColormap.from_list('custom',horizon_colors)
solar_extra = LinearSegmentedColormap.from_list('custom',solar_extra_colors)

palette = [
    '#533f7c',                        # LUM
    '#37888e',                        # NRP
    '#4c8d49',                        # SQM
    '#ce8d24',                        # MES
    '#aa4743',                        # NEC
]

BuGn9 = ["#f7fcfd","#e5f5f9","#ccece6","#99d8c9","#66c2a4","#41ae76","#238b45","#006d2c","#00441b"]
BuGn_new = ["#f7fcfd","#04702F","#006227","#005321","#00441B"]
BuGn_new_cmap = LinearSegmentedColormap.from_list('custom',BuGn_new)

## Load data

In [ ]:
adata = sc.read("Reproducibility/Data/DOGMA/UC_DOGMA_RNA_ATAC_ADT_after_QC.h5ad")

fig_dir = "Reproducibility/Results/Plots/Global/"
os.makedirs(fig_dir, exist_ok = True)

In [ ]:
mapping = {
    "UC": "Epithelial",
    "NEC": "Epithelial",
    "Pericyte": "MSC",
    "CAF": "MSC",
}

s = adata.obs["coarse_celltype"]

if pd.api.types.is_categorical_dtype(s):
    s2 = s.cat.add_categories(["Epithelial", "MSC"]).replace(mapping).cat.remove_unused_categories()
else:
    s2 = s.replace(mapping)

adata.obs["coarse_celltype2"] = s2.astype("category")

cat_type = CategoricalDtype(categories=["CD4_Tconv","Treg","CD8_T","NK_ILC","B","Mono_Mac","DC","Mast","Epithelial","MSC","Endothelial"], ordered=True)
adata.obs["coarse_celltype2"] = adata.obs["coarse_celltype2"].astype(cat_type)

In [ ]:
adata.obs['STAGE2'] = np.where(
    adata.obs['Organ'] == 'UTUC',  # Condition
    'UTUC',                        # Value if True
    adata.obs['STAGE']             # Value if False
)

mapping = {'Early_L': 'NMIBC_L', 'Early_H': 'NMIBC_H', 'Advanced': 'MIBC'}

s = adata.obs['STAGE2']
if pd.api.types.is_categorical_dtype(s):
    s2 = s.replace(mapping).cat.remove_unused_categories()
else:
    s2 = s.replace(mapping).astype('category')

adata.obs['STAGE2'] = s2

order = ['NMIBC_L', 'NMIBC_H', 'MIBC', 'UTUC', 'post_BCG']
adata.obs['STAGE2'] = adata.obs['STAGE2'].cat.reorder_categories(order, ordered=True)

In [ ]:
cat_type = CategoricalDtype(categories=[
    'CD4_Tn','CD4_Tcm','CD4_Tsen','CD4_T_CD26','CD4_CTL','CD4_Th17', 
    'CD4_Tfh-like','CD4_Tph-like','CD4_T_proliferative','Treg_naive','Treg_effector',
    'CD8_Tn','CD8_Tcm','CD8_Tem','CD8_Temra','CD8_Trm','CD8_Tex_1','CD8_Tex_2','CD8_T_proliferative',
    'NK_CD56_CD49a_Hi_CD103_Hi','NK_CD56_CD49a_Hi_CD103_Lo','NK_CD56_CD49a_Lo','NK_CD56_dim','ILC3','MAIT',
    'B_naive','B_memory','Atypical_B','GC_B','Plasma',
    'Mono','MDSC-like','TAM_TREM2','TAM_FOLR2',
    'cDC1','cDC2','mregDC','preDC','pDC','Mast',
    'Normal','LUM','NRP','SQM','MES','NEC',
    'proCAF','iCAF_CD321','iCAF_SLC14A1','matCAF','matCAP','contCAP','vSMC',
    'Endothelial'], ordered=True)

adata.obs["celltype"] = adata.obs["celltype"].astype(cat_type)

## Visualization

### UMAP

In [ ]:
# Fig.1B
sc.set_figure_params(figsize=(3, 3))
sc.set_figure_params(vector_friendly=True, dpi_save=100)
sc.pl.umap(
    adata,
    color=["coarse_celltype2"],
    frameon=False,
    ncols=1,
    size=0.3,
    palette=["#8dd3c7","#ffffb3","#bebada","#fb8072","#80b1d3","#fdb462","#b3de69","#fccde5","#1B9E77","#D95F02","#E7298A"],   # Set3 + Dark2
#    legend_loc="on data",
    legend_fontsize=7,
    show = False
)
plt.savefig(f"{fig_dir}Figure1B.pdf", bbox_inches='tight')
plt.close()

In [ ]:
# Fig.S1C
sc.set_figure_params(figsize=(3, 3))
sc.set_figure_params(vector_friendly=True, dpi_save=100)
sc.pl.umap(
    adata,
    color=["celltype"],
    frameon=False,
    ncols=1,
    size=0.3,
    legend_fontsize=7,
    show = False
)
plt.savefig(f"{fig_dir}FigureS1C.pdf", bbox_inches='tight')
plt.close()

In [ ]:
# Fig.1D
sc.set_figure_params(figsize=(3, 3))
sc.set_figure_params(vector_friendly=True, dpi_save=100)
sc.pl.umap(
    adata,
    color=["sample"],
    frameon=False,
    ncols=1,
    size=0.3,
    legend_fontsize=7,
    show = False
)
plt.savefig(f"{fig_dir}Figure1D_patients.pdf", bbox_inches='tight')
plt.close()

In [ ]:
# Fig.1D  
import scvelo as scv
scv.set_figure_params(vector_friendly=True, dpi_save=100)
scv.pl.scatter(adata, 
               dpi = 300,
               groups=[[c] for c in ["NMIBC_L","NMIBC_H","MIBC","UTUC","post_BCG"]], 
               color='STAGE2', 
               frameon=False,
               ncols=5,
               size = 0.3,
               linewidth = 0,
               outline_width = 0,
               figsize = (3,3),
               show = False
               )
plt.savefig(f"{fig_dir}Figure1D_split_by_group.pdf", bbox_inches='tight')
plt.close()

### RNA dotplot

In [ ]:
adata_RNA = adata[:, adata.var.modality == "Gene Expression"].copy()
adata_RNA.layers["counts"] = adata_RNA.X.copy()
sc.pp.normalize_total(adata_RNA)
sc.pp.log1p(adata_RNA)

In [ ]:
marker_genes = ["PTPRC", "CD3D","CD4", "IL2RA", "CTLA4", "TNFRSF9", "TIGIT", "CD177",  
                "CD8A", "KLRK1", "KIR3DL1", "KLRD1", "NCR1",  
                "CD19", "CR2", "CD22", "IGHM", "IGHD", "TNFRSF13C",  
                "ITGAX", "CD14", "ITGAM", "FCGR1A", "HLA-DRA", "CD1C", "CLEC4C", "IL3RA", "KIT",  
                "EPCAM", "NCAM1", "PDPN", "PDGFRA", "THY1", "MCAM", "PECAM1", "CD34", "ENG"]

In [ ]:
# Fig.S1D
sc.pl.dotplot(adata_RNA, 
              marker_genes, 
              groupby= 'coarse_celltype', 
              dendrogram = False,
              standard_scale="var",
              show = False)
plt.savefig(f"{fig_dir}FigureS1D.pdf", bbox_inches='tight')
plt.close()

### Protein heatmap

In [ ]:
from muon import prot as pt
adt_df = adata_RNA.obsm["protein_expression"].copy()
adt_df.columns = [col.split("-")[-1] for col in adt_df.columns]

pro_adata = sc.AnnData(adt_df, obs=adata_RNA.obs)
pro_adata.layers["counts"] = pro_adata.X.copy()
pro_adata.obsm["X_umap"] = adata.obsm["X_umap"]
pro_adata.obs = adata.obs

pt.pp.clr(pro_adata)

In [ ]:
marker_proteins  = ["CD45","CD3",
                    'CD4','CD25','CTLA4','41BB','TIGIT','CD177',
                    'CD8','NKG2D','NKB1','CD94','NKp46',
                    'CD19','CD21',"CD22",'IgM','IgD','BAFFR',
                    'CD11c','CD14','CD11b','CD64','HLADR','CD1c','BDCA2','CD123',
                    'ckit',
                    'EpCAM','NCAM',
                    'Podoplanin','PDGFRa','Thy1','CD146',
                    'CD31','CD34','CD105'
                    ]

In [ ]:
sc.pl.matrixplot(pro_adata, 
                 marker_proteins, 
                 groupby = "coarse_celltype",
                 dendrogram=False,
                 standard_scale='var',                 
                 cmap='plasma',
                 show = False)
plt.savefig(f"{fig_dir}Figure1C.pdf", bbox_inches='tight')
plt.close()

### QC plots

In [ ]:
adata.obs["log10_UMI_RNA"] = np.log10(adata.obs["total_counts_RNA"])
adata.obs["log10_Genes_RNA"] = np.log10(adata.obs["n_genes_RNA"])
adata.obs["log10_total_fragment_counts"] = np.log10(adata.obs["total_counts_ATAC"])

In [ ]:
sc.set_figure_params(figsize=(3, 3))
sc.set_figure_params(vector_friendly=True, dpi_save=100)
sc.pl.umap(
    adata,
    color=['log10_UMI_RNA', 'log10_Genes_RNA','pct_counts_mt',"log10_total_fragment_counts"],
    frameon=False,
    ncols=2,
    size=0.3,
    cmap = "viridis",
    legend_fontsize=7,
    show = False
)
plt.savefig(f"{fig_dir}FigureS1E.pdf", bbox_inches='tight')
plt.close()

### Proportion bar plots

In [ ]:
import sys
sys.path.append(os.path.abspath('Reproducibility/Scripts/Source/utility_functions'))
from scRNAseq_Visualization_Functions import *

In [ ]:
adata.obs['is_cell'] = 'cell'
cat_type = CategoricalDtype(categories=["cell","dummy"], ordered=True)
adata.obs['is_cell'] = adata.obs['is_cell'].astype(cat_type)

In [ ]:
adata_immune = adata[adata.obs['lineage'].isin(['CD4_T','CD8_T_NK_ILC','B','Myeloid'])].copy()
adata_nonimmune = adata[adata.obs['lineage'].isin(['Epithelial','MSC','Endothelial'])].copy()

Immune

In [ ]:
pts_order = ['BC_018','UTUC_006','BC_022','BC_026','BC_012','BC_019','BC_050','BC_017','BC_024','BC_021',
             'BC_014','BC_011','BC_042','BC_041','BC_016','BC_037','BC_023','BC_008','BC_033',
             'BC_020','BC_030','BC_013','BC_031',"UTUC_003","UTUC_001",'BC_028','BC_010','BC_029','BC_038',
             'BC_036',"UTUC_005",'BC_045',"UTUC_007","UTUC_004",
             'BC_032','BC_040','BC_043','BC_048','BC_027','BC_047','BC_039','BC_044']

In [ ]:
from palettable.cartocolors.qualitative import Pastel_10
colors = Pastel_10.mpl_colors
filtered_colors = [color for color in colors if not (abs(color[0] - color[1]) < 0.1 and abs(color[1] - color[2]) < 0.1)]
filtered_cmap = plt.cm.colors.ListedColormap(filtered_colors)

In [ ]:
plot_composition_simple(adata_immune, x = 'sample', y ='coarse_celltype', 
                        x_order=pts_order, 
                        y_order=["CD4_Tconv","Treg","CD8_T","NK_ILC",'B','Mono_Mac','DC','Mast'], norm=True, style='bar', 
                        bar_figsize=(8, 6), 
                        cmap=filtered_cmap, 
                        bar_width=0.6, bar_linkage=True, bar_link_alpha=0.5, 
                        heat_figsize=(5, 5), only_size=False, heat_size_scale=50, heat_marker='.', 
                        return_df=True, melt=False, return_df_annot=None)
plt.savefig(f"{fig_dir}Figure1E_immune_prop.pdf", bbox_inches='tight')
plt.close()

In [ ]:
features = ['STAGE', 'Organ']
plot_composition_complex_log(adata_immune, 
                             x = 'sample', 
                             y = 'is_cell', 
                             x_order=pts_order, 
                             y_order=['cell'],
                             x_annot=features, 
                             xticklabels=True,
                             figsize=(10, 8), 
                             colors=None, 
                             palette=palettable.cartocolors.sequential.Mint_7_r.get_mpl_colormap(), \
                             bar_x_norm=False, 
                             bar_width=0.6, bar_x_linkage=True, bar_x_link_alpha=0.5, \
                             heat_size_scale=50, heat_marker='.')
plt.savefig(f"{fig_dir}Figure1E_immune_counts.pdf", bbox_inches='tight')
plt.close()

Non immune

In [ ]:
pts_order = ["BC_012","BC_024","BC_019","BC_026","BC_018","BC_017","BC_022","BC_050","UTUC_006","BC_021",
             "BC_011","BC_014","BC_042","BC_037","BC_033","BC_023","BC_016","BC_041","BC_008",  
             "BC_045","BC_030","UTUC_005","BC_020","BC_010","BC_036","BC_038","BC_013","UTUC_001",
             "UTUC_007","BC_031","UTUC_003","UTUC_004","BC_028","BC_029"]

In [ ]:
from palettable.cartocolors.qualitative import Prism_10_r
colors = Prism_10_r.mpl_colors
filtered_colors = [color for color in colors if not (abs(color[0] - color[1]) < 0.1 and abs(color[1] - color[2]) < 0.1)]
filtered_cmap = plt.cm.colors.ListedColormap(filtered_colors)

In [ ]:
plot_composition_simple(adata_nonimmune, x = 'sample', y ='coarse_celltype2', 
                        x_order=pts_order, 
                        y_order=["Epithelial",'MSC','Endothelial'], norm=True, style='bar', 
                        bar_figsize=(8, 6), 
                        cmap=palettable.cartocolors.qualitative.Prism_9_r.get_mpl_colormap(), 
                        bar_width=0.6, bar_linkage=True, bar_link_alpha=0.5, 
                        heat_figsize=(5, 5), only_size=False, heat_size_scale=50, heat_marker='.', 
                        return_df=True, melt=False, return_df_annot=None)
plt.savefig(f"{fig_dir}Figure1E_non-immune_prop.pdf", bbox_inches='tight')
plt.close()

In [ ]:
features = ['STAGE', 'Organ']
plot_composition_complex_log(adata_nonimmune, 
                             x = 'sample', 
                             y = 'is_cell', 
                             x_order=pts_order, 
                             y_order=['cell'],
                             x_annot=features, 
                             xticklabels=True,
                             figsize=(10, 8), 
                             colors=None, 
                             palette=palettable.cartocolors.sequential.Mint_7_r.get_mpl_colormap(), \
                             bar_x_norm=False, 
                             bar_width=0.6, bar_x_linkage=True, bar_x_link_alpha=0.5, \
                             heat_size_scale=50, heat_marker='.')
plt.savefig(f"{fig_dir}Figure1E_non-immune_counts.pdf", bbox_inches='tight')
plt.close()